In [ ]:
from pyspark.ml.feature import VectorAssembler
from xgboost.spark import SparkXGBClassifier, SparkXGBClassifierModel
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc


class XGBoostModel:
    """
    XGBoost Model for Loan Prediction (Spark Version)
    """

    def __init__(self):
        self.model = None

        self.feature_names = [
            "Age", "Income", "LoanAmount", "Credit_Score",
            "Employment_Years", "Credit_History", "Has_Defaulted",
            "Dependents", "Gender", "Education_Level",
            "Married", "Job_Type", "Property_Area"
        ]

    # ─────────────────────────────
    # 1. Feature Engineering
    # ─────────────────────────────
    def build_features(self, df):

        df = df.withColumnRenamed("Loan_Status", "label")

        assembler = VectorAssembler(
            inputCols=self.feature_names,
            outputCol="features",
            handleInvalid="keep"
        )

        df = assembler.transform(df)

        print(" Features built successfully")
        return df

    # ─────────────────────────────
    # 2. Train/Test Split
    # ─────────────────────────────
    def split_data(self, df, ratio=0.8):

        train_df, test_df = df.randomSplit([ratio, 1 - ratio], seed=42)

        print(f"Train: {train_df.count()} rows")
        print(f"Test: {test_df.count()} rows")

        return train_df, test_df

    # ─────────────────────────────
    # 3. Create Model
    # ─────────────────────────────
    def create_model(self):

        return SparkXGBClassifier(
            features_col="features",
            label_col="label",
            prediction_col="prediction",
            probability_col="probability",
            max_depth=6,
            eta=0.1,
            num_round=100
        )

    # ─────────────────────────────
    # 4. Train Model
    # ─────────────────────────────
    def train_model(self, train_df):

        xgb = self.create_model()

        print(" Training XGBoost model...")

        self.model = xgb.fit(train_df)

        print(" Training complete!")

        return self.model

    # ─────────────────────────────
    # 5. Predict
    # ─────────────────────────────
    def predict(self, test_df):

        print(" Making predictions...")

        predictions = self.model.transform(test_df)

        print(" Predictions done!")

        return predictions

    # ─────────────────────────────
    # 6. Evaluation
    # ─────────────────────────────
    def evaluate(self, predictions):

        acc = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="accuracy"
        ).evaluate(predictions)

        f1 = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="f1"
        ).evaluate(predictions)

        auc_score = BinaryClassificationEvaluator(
            labelCol="label",
            rawPredictionCol="rawPrediction",
            metricName="areaUnderROC"
        ).evaluate(predictions)

        precision = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="weightedPrecision"
        ).evaluate(predictions)

        recall = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="weightedRecall"
        ).evaluate(predictions)

        return acc, f1, auc_score, precision, recall

    # ─────────────────────────────
    # 7. Print Results
    # ─────────────────────────────
    def print_evaluation(self, predictions):

        acc, f1, auc_score, precision, recall = self.evaluate(predictions)

        print("\n XGBOOST - RESULTS")
        print("=" * 40)
        print(f"Accuracy : {acc:.4f}")
        print(f"F1 Score : {f1:.4f}")
        print(f"AUC      : {auc_score:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall   : {recall:.4f}")

        return acc, f1, auc_score, precision, recall
    #________________________________________
    # Importance
    #________________________________________
    def get_feature_importance_df(self):

    importance = self.model.get_booster().get_score(importance_type='gain')

    fi = pd.DataFrame({
        "feature": self.feature_names,
        "importance": [importance.get(f"f{i}", 0.0) for i in range(len(self.feature_names))]
    })

    fi["importance_pct"] = fi["importance"] / fi["importance"].sum() * 100

    fi = fi.sort_values(by="importance_pct", ascending=False)

    return fi

    # ─────────────────────────────
    # 8. Confusion Matrix
    # ─────────────────────────────
    def plot_confusion_matrix(self, predictions):

        pred_pd = predictions.select("label", "prediction").toPandas()

        cm = pd.crosstab(pred_pd["label"], pred_pd["prediction"])

        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.title("XGBoost - Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.show()

        return cm

    # ─────────────────────────────
    # 9. ROC Curve
    # ─────────────────────────────
    def plot_roc_curve(self, predictions):

        pred_pd = predictions.select("label", "probability").toPandas()

        y_true = predictions.select("label").toPandas()["label"]
        y_score = pred_pd["probability"].apply(lambda x: float(x[1]))

        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)

        plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
        plt.plot([0, 1], [0, 1], '--')
        plt.title("ROC Curve - XGBoost")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.legend()
        plt.show()

        return roc_auc

    # ─────────────────────────────
    # 10. Feature Importance
    # ─────────────────────────────
    def plot_feature_importance(self):

    importance = self.model.get_booster().get_score(importance_type='gain')

    fi = pd.DataFrame({
        "feature": self.feature_names,
        "importance": [importance.get(f"f{i}", 0.0) for i in range(len(self.feature_names))]
    })

    fi = fi.sort_values(by="importance_pct", ascending=False)

    sns.barplot(x="importance", y="feature", data=fi)
    plt.title("XGBoost Feature Importance")
    plt.show()

    return fi

    # ─────────────────────────────
    # 11. Save Model
    # ─────────────────────────────
    def save_model(self, path="xgboost_model"):

        self.model.write().overwrite().save(path)

        print(f" Model saved at: {path}")



"""
# Initialize XGBOOST Model
xgb_model = XGBoostModel()

# Build features
df_ready = xgb_model.build_features(df)

# Split data
train_df, test_df = xgb_model.split_data(df_ready)

# Train model
xgb_model.train_model(train_df)

# Make predictions
predictions = xgb_model.predict(test_df)

# Evaluate and print results
xgb_model.print_evaluation(predictions)

# Visualizations
xgb_model.plot_confusion_matrix(predictions)
xgb_model.plot_roc_curve(predictions)
xgb_model.plot_feature_importance()

# Save model
xgb_model.save_model()
"""